In [1]:
import pandas as pd
from geostat_isoscapes_tools import geostat_utils as gutils, plot_utils as putils, sisal_utils as sutils, utils
import matplotlib.pyplot as plt 
import numpy as np
import skgstat as skg
from tqdm import tqdm
from scipy.spatial.distance import pdist
import seaborn as sns
import json

sns.set_style('dark')

In this notebook, we compare variograms of itrace and sisal data. In particular, we select itrace data with the union of circles centered around sisal points at each time slice.

At the end of the day, the goal is to develop a function that would automatically define the mask, compute sisal and itrace variograms and save the results.

In [2]:
# Load sisal data 
sisal_df = gutils.get_sisal_data_for_kriging(res=200,countries=None,verbose=False)

# load itrace data
itrace_df = gutils.preprocessed_itrace_data(res=200*365, verbose=False)

loading sisal data
sisal dataframe is ready
loading itrace files
itrace DataFrame is ready


In [3]:
itrace_df.columns

Index(['time', 'latitude', 'longitude', 'd18Op'], dtype='object')

In [4]:
sisal_df.columns

Index(['mineralogy', 'age', 'age_method', 'age_uncert_pos', 'age_uncert_neg',
       'entity_id', 'sample_id', 'site_id', 'site_name', 'longitude',
       'latitude', 'd18O_measurement', 'd18O_precision', 'T_linear',
       'T_nearest', 'T', 'd18Op_VSMOW_exactconv', 'd18Op_VSMOW_linearized',
       'd18Oc_VSMOW', 'binned_age'],
      dtype='object')

In [5]:
# isolate sisal points locations 
unique_locs = sisal_df[['latitude','longitude']].drop_duplicates()
 

In [6]:
from scipy.spatial.distance import cdist

def haversine(u, v):
    """Haversine distance (km) between two (lat,lon) points."""
    lat1, lon1 = np.radians(u)
    lat2, lon2 = np.radians(v)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * 6371 * np.arcsin(np.sqrt(a)) # since earth radius = approx 6371 km

# SISAL center points
sisal_locs = unique_locs.values  # shape (n_sisal, 2)

# Global data points
global_locs = itrace_df[['latitude','longitude']].values  # shape (n_global, 2)

# Compute min distance from each global point to nearest SISAL point
distances = cdist(global_locs, sisal_locs, metric=haversine)  # shape (n_global, n_sisal)
min_distances = distances.min(axis=1)  # shape (n_global,)

# Mask: True where within radius
radius_km = 1500
mask = min_distances <= radius_km

# Apply mask
itrace_masked = itrace_df[mask]

In [7]:
putils.plot_global_map(data=itrace_masked,
                    quantity_col='d18Op',
                    title = 'masked itrace data',
                    quantity='d18Op',
                    unit='per mille (VSMOW)',
                    proj=False,
                    symbol='circle')